
# SOLUCIÓN al TP N° 5- De Texto Caótico a Dataset Limpio
Objetivo
Construir un mini-pipeline de preprocesamiento que transforme texto libre en un dataset estructurado listo para
análisis.



```
Para que funcione, es fundamental que la estructura de directorios en su entorno local coincida con la siguiente:

- `logs/` (para almacenar los logs del sistema)
- `data/raw/ingesta_cruda.txt` (el archivo de entrada con los datos crudos)
- `data/processed/` (donde se guardarán los datos limpios y estructurados en JSON y CSV)
- `data/problems/` (donde se almacenarán los registros con errores para su revisión)

Instrucciones:

1.  Crear directorios: Antes de ejecutar el script, asegúrense de que las carpetas `logs/`, `data/raw/`, `data/processed/` y `data/problems/` existan en la raíz de su proyecto. Si no existen, créenlas manualmente o modifiquen el código.
2. Archivo de entrada: Coloquen el archivo `ingesta_cruda.txt` dentro de la carpeta `data/raw/`.

```

# Ingesta
Lectura de texto crudo para su posterior procesamiento. Esta etapa es fundamental para obtener los datos con los que trabajaremos.

In [ ]:
#contenido del archivo
'''
### INICIO DE LOTE - SISTEMA DE INGESTA V2.4 ###
ID: 001 - Cliente: Ana Lopez - Tel: 351-4567890 - Mail: ANA@GMAIL.COM
Juan Perez | 351 555 1234 | juan_perez88@hotmail.com
Registro Corrupto ##### Error 404 #####
Cli: Maria Garcia - Tel: 3514567788 - Mail: mgarcia@yahoo.com.ar
Carlos Gomez sin telefono - Mail: cgomez@empresa.com
    Contacto: Laura Mendez - 351-abc-1234 - laura@mail.com
ID: 002 - Cliente: Ana Lopez - Tel: 351-4567890 - Mail: ANA@GMAIL.COM
Sofia Martinez ; 3512223344 ; sofia.m@gmail.com
PEDRO ALVAREZ / 3514445555 / p.alvarez@outlook.com
Cliente Nuevo: Lucia Torres - Cel: (351) 111 2222 - lutorres@gmail.com
--------------------------------------------------
ID: 003 - Roberto Gomez - Tel: 3516667777 - Mail: robertog@gmail
Marta Sanchez , 351-999-8888 , marta.sanchez@uol.com.br
### LINEA DE SISTEMA - LOG 2024-03-09 ###
Jorge Rodriguez | tel 351.444.1111 | jrodriguez@gmail.com
Cli: "Elena Paz" - Tel: 351-555-0000 - Mail: epaz@proyectos.org
Ricardo Fort - Cel: 351 000 1122 - ricky@gmail.com
Socio: Beatriz Pinzon - 351-999-0000 - b.pinzon@eco-moda.com
ID: 004 - Cliente: Juan Perez - Tel: 3515551234 - Mail: juan_perez88@hotmail.com
fede_tolosa - 3514545454 - federico@tolosa.com.ar
Esteban Quito / Tel: 123 / equito@gmail.com
ID: 005 - Valentina Flores - Tel: 351-777-6655 - Mail: vflores@gmail.com
Monica Galindo | 351 222 1111 | moni.g@hotmail.com
GUSTAVO CERATI - 351 444 0000 - soda@stereo.com
Cli: Mariana Enríquez - Tel: 351-666-3333 - Mail: m.enriquez@letras.ar
Dato Incompleto - Cliente: Pablo Picasso - Tel: 351-1234567
### FIN DE ARCHIVO - TOTAL REGISTROS: 25 ###
'''


In [ ]:
#configuramos logging
logging.basicConfig(
    filename='sistema.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [ ]:
#no es un archivo csv - modo lectuta y codificacion
def leer_archivo(ruta):
    try:
        logging.info(f"Intentando leer el archivo: {ruta}")
        with open(ruta, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        logging.error("Archivo no encontrado")
        return ""
    except UnicodeError:
        logging.error("Error de codificación")
        return ""
    except Exception as e:
        logging.error(f"Error inesperado: {e}")
        return ""

# Parsing
Esta etapa se encarga de dividir el texto crudo en líneas individuales, eliminando espacios en blanco innecesarios y líneas vacías para facilitar el procesamiento posterior.

In [ ]:
#parseo
def parsear(texto):
    return [linea.strip() for linea in texto.split("\n") if linea.strip()]

#Extracción flexible: Primero capturo todo lo posible, después decido qué sirve
- Captura más casos
- Tolera ruido
- No pierde información útil

In [ ]:
#extraer informacion
def extraer_nombre(registro):
    limpio = re.sub(r'^(ID: \d+ - |Cliente: |Cli: |Socio: |Contacto: |Cliente Nuevo: )', '', registro)
    res = re.search(r'[A-Za-zÁÉÍÓÚÑáéíóúñ]+(?:\s[A-Za-zÁÉÍÓÚÑáéíóúñ]+)+', limpio)
    return res.group() if res else None

def extraer_mail(registro):
    res = re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', registro)
    return res.group() if res else None

def extraer_telefono(registro):
    res = re.search(r'(\+?\d{1,3}[\s-]?)?(\(?\d{2,4}\)?[\s-]?)?\d{3,4}[\s-]?\d{4}', registro)
    return res.group() if res else None

# Normalizacion
Esta etapa estandariza los datos extraídos para asegurar consistencia y uniformidad, haciendo que sean comparables y utilizables en análisis posteriores. Por ejemplo, convierte teléfonos a un formato único y mails/nombres a minúsculas.

In [ ]:
#normalizacion
def normalizar_tel(tel):
    if not tel:
        return None
    tel = re.sub(r'\D', '', tel)
    if tel.startswith('549'):
        tel = tel[3:]
    elif tel.startswith('54'):
        tel = tel[2:]
    if len(tel) > 10 and tel.startswith('9'):
        tel = tel[1:]
    if len(tel) == 10:
        return tel
    return None

In [ ]:
def normalizar_nombre(nombre):
  return nombre.strip().lower()

def normalizar_mail(mail):
  return mail.strip().lower()

#Validacion más estricta
Decidís:

- válido
- inválido
- corregible

In [ ]:
#validaciones
def validar_nombre(nombre):
    if not nombre: return False
    return bool(re.match(r'^[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(\s[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+)+$', nombre))

def validar_mail(mail):
    if not mail: return False
    return bool(re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', mail))

# Estructuración - datos limpios
Esta etapa organiza los datos limpios y validados en un formato estructurado, generalmente una lista de diccionarios, para facilitar su análisis y uso posterior.

Orden de limpieza:

extraer → normalizar → validar → deduplicar → exportar

In [ ]:
#estructurar en lista de diccionarios
#usaremos un conjunto para evitar duplicados
def procesar_registro(registros):
    errores = []
    validos = []
    vistos=set()
    for r in registros:
        # 1. Ignorar líneas de sistema (Headers/Logs) antes de procesar
        if r.startswith("###") or r.startswith("---") or "SISTEMA" in r:
            continue

        problemas = []
        nombre = extraer_nombre(r)
        mail = extraer_mail(r)
        mail= mail.lower() if mail else None
        tel = extraer_telefono(r)

        if not tel:
            problemas.append("Teléfono no encontrado")
        else:
            tel = normalizar_telefono(tel)
            if not tel:
                problemas.append("Teléfono inválido")

        # Validación de Nombre
        if not nombre:
            problemas.append("Nombre no encontrado")
        elif not validar_nombre(nombre):
            problemas.append(f"Nombre inválido format: {nombre}")

        # Validación de Mail
        if not mail:
            problemas.append("Mail no encontrado")
        elif not validar_mail(mail):
            problemas.append(f"Mail inválido format: {mail}")


        if problemas:
            errores.append({
                'registro': r,
                'nombre': nombre,
                'mail': mail,
                'telefono': tel,
                'error': problemas
            })
            logging.warning(f"Registro con problemas: {problemas} -> {r}")
        else:
            r_unico = (nombre.strip().lower(), mail, tel)
            if r_unico not in vistos:
                validos.append({
                    'nombre': nombre.title(),       #aprovecho a normalizar
                    'mail': mail,
                    'telefono': tel
                })
                vistos.add(r_unico)
            else:
                logging.warning(f"Registro duplicado omitido: {nombre}")
    return errores, validos

# Exportación
Esta etapa se encarga de guardar los datos procesados en formatos accesibles, como JSON o CSV, y de almacenar los registros con errores para su posterior revisión.
Luego podremos hacer un análisis de los errores.

In [ ]:
#exportacion dual
def exportar_dual(validos):
    if not validos: return

    # Crear carpetas si no existen
    os.makedirs("data/processed", exist_ok=True)

    with open("data/processed/clientes.json", "w", encoding='utf-8') as f:
        json.dump(validos, f, indent=4, ensure_ascii=False)

    with open("data/processed/clientes.csv", "w", encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['nombre', 'mail', 'telefono'])
        writer.writeheader()
        writer.writerows(validos)


def exportar_errores(errores):
    if not errores: return
    os.makedirs("data/problems", exist_ok=True)
    with open("data/problems/descartes.json", "w", encoding='utf-8') as f:
        json.dump(errores, f, indent=4, ensure_ascii=False)


In [ ]:
#main
def main():
    logging.info("Comienza el proceso de ingesta")
    ruta_input = "data/raw/ingesta_cruda.txt"
    if not os.path.exists(ruta_input):
        logging.error(f"No existe el archivo de entrada en {ruta_input}")
        return

    texto = leer_archivo(ruta_input)
    registros = parsear(texto)

    logging.info(f"Registros encontrados: {len(registros)}")

    errores, validos = procesar_usuarios(registros)

    exportar_dual(validos)
    exportar_errores(errores)

    logging.info(f"Proceso finalizado. Validos: {len(validos)}, Errores: {len(errores)}")

#Script final para probar.
Recurden mi estructura de directorios:
- `logs/`
- `data/raw/ingesta_cruda.txt`
- `data/processed/`
- `data/problems/`


In [ ]:
import logging
import re
import csv
import json
import os

logging.basicConfig(
    filename="logs/sistema.log",
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
#verificamos directorios
def asegurar_directorios():
    os.makedirs("logs", exist_ok=True)
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/problems", exist_ok=True)

#leer archivo
def leer_archivo(ruta):
    try:
        with open(ruta,'r',encoding='utf-8' ) as f:
            return f.read()
    except FileNotFoundError:
        logging.error("Archivo no encontrado")
        return ""
    except UnicodeError:
        logging.error("Error de codificación")
        return ""
    except Exception as e:
        logging.error(f"Error inesperado {e}")
        return ""

#parsear
def parsear(texto):
    return [linea.strip() for linea in texto.split('\n') if linea.strip()]
#extraer informacion
def extraer_nombre(registro):
    limpio = re.sub(r'^(ID: \d+ - |Cliente: |Cli: |Socio: |Contacto: |Cliente Nuevo: )', '', registro)
    res = re.search(r'[A-Za-zÁÉÍÓÚÑáéíóúñ]+(?:\s[A-Za-zÁÉÍÓÚÑáéíóúñ]+)+', limpio)
    return res.group() if res else None

def extraer_mail(registro):
    res = re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', registro)
    return res.group() if res else None

def extraer_telefono(registro):
    res = re.search(r'(\+?\d{1,3}[\s-]?)?(\(?\d{2,4}\)?[\s-]?)?\d{3,4}[\s-]?\d{4}', registro)
    return res.group() if res else None

#normalizar
def normalizar_telefono(tel):
    if not tel:
        return None
    tel=re.sub(r'\D','',tel)
    if tel.startswith('549'):
        tel=tel[3:]
    if tel.startswith('54'):
        tel=tel[2:]
    if len(tel)>10 and tel.startswith('9'):
        tel=tel[1:]
    if len(tel)==10:
        return tel
    else:
        return None

#validar información
def validar_nombre(nombre):
    if not nombre: return False
    return bool(re.match(r'^[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(\s[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+)+$', nombre))

def validar_mail(mail):
    if not mail: return False
    return bool(re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', mail))

#estructurar en lista de diccionarios
def procesar_registros(registros):
    errores = []
    validos = []
    vistos=set()
    for r in registros:
        # 1. Ignorar líneas de sistema (Headers/Logs) antes de procesar
        if r.startswith("###") or r.startswith("---") or "SISTEMA" in r:
            continue

        problemas = []
        nombre = extraer_nombre(r)
        mail = extraer_mail(r)
        mail= mail.lower() if mail else None
        tel = extraer_telefono(r)

        if not tel:
            problemas.append("Teléfono no encontrado")
        else:
            tel = normalizar_telefono(tel)
            if not tel:
                problemas.append("Teléfono inválido")

        # Validación de Nombre
        if not nombre:
            problemas.append("Nombre no encontrado")
        elif not validar_nombre(nombre):
            problemas.append(f"Nombre inválido format: {nombre}")

        # Validación de Mail
        if not mail:
            problemas.append("Mail no encontrado")
        elif not validar_mail(mail):
            problemas.append(f"Mail inválido format: {mail}")


        if problemas:
            errores.append({
                'registro': r,
                'nombre': nombre,
                'mail': mail,
                'telefono': tel,
                'error': problemas
            })
            logging.warning(f"Registro con problemas: {problemas} -> {r}")
        else:
            r_unico = (nombre.strip().lower(), mail, tel)
            if r_unico not in vistos:
                validos.append({
                    'nombre': nombre.title(),       #aprovecho a normalizar
                    'mail': mail,
                    'telefono': tel
                })
                vistos.add(r_unico)
            else:
                logging.warning(f"Registro duplicado omitido: {nombre}")
    return errores, validos

#exportar dual
def exportar_dual(validos):
    with open("data/processed/clientes.json","w",encoding='utf-8') as f:
        json.dump(validos,f,indent=4,ensure_ascii=False)
    with open("data/processed/clientes.csv","w",newline="", encoding='utf-8') as f:
        writer=csv.DictWriter(f, fieldnames=["nombre","mail", "telefono"])
        writer.writeheader()
        writer.writerows(validos)

#exportacion de descartados
def exportar_errores(errores):
    with open("data/problems/descartados.json","w",encoding='utf-8') as f:
        json.dump(errores,f,indent=4,ensure_ascii=False)


#main
def main():
    logging.info("Comienza el proceso de ingesta")
    asegurar_directorios()
    ruta_input = "data/raw/ingesta_cruda.txt"
    if not os.path.exists(ruta_input):
        logging.error(f"No existe el archivo de entrada en {ruta_input}")
        return

    texto = leer_archivo(ruta_input)
    registros = parsear(texto)

    logging.info(f"Registros encontrados: {len(registros)}")

    errores, validos = procesar_registros(registros)

    exportar_dual(validos)
    exportar_errores(errores)

    logging.info(f"Proceso finalizado. Validos: {len(validos)}, Errores: {len(errores)}")

if __name__=='__main__':
    main()